# Question 1: How Can We Identify SMB Sellers

## Assignment
> In our businesses we have always seen it as a strength that registration is very low effort. You only need an email address and a phone number to start trading on our platforms.
>
> The downside of this is that we do not have a full customer view of our users. Especially in our SYI flow, we do not know much about our sellers. We do, however, know a lot about what happens on our platforms. We can easily understand the behavior of our visitors; buyers, sellers, and browsers.
>
> In the first attachment, a list of information can be found that we do have on our sellers and the activity. This data goes three years back.
>
> It is December 2024. We want to introduce SMB bundles, which will make it easier for SMB sellers to use our platform to reach our enormous customer base. To launch this, a big task is to identify which sellers are actual business sellers and which are consumers.
>
> We cannot simply call all 10+ million users to ask this, and using a pop-up on our platform is not expected to bring reliable results. We need to narrow down this base as good as we can using the existing data.
>
> Using the description of the available data, think of identifiers that point towards a seller being SMB, and argue why you believe these could be good indicators. Combine information from the various sources to think of 3-8 good identifiers. (For example, “Every seller that has listed at least 2 advertisements in the category ‘Tuin en Terras’ (Garden and terrace) is likely to be a business seller”.)


## SMB Identification Methods

Use the available schemas to create seller-level indicators. Each method is a separate signal; the final score combines them.


### 1. General Seller Activity

**Signal:** sellers that repeatedly list and stay active look more business-like than one-off sellers.

Schemas: **User Related Info**, **Ad Related Info**.

```mermaid
flowchart LR
    user["User Related Info<br/>User id"]
    ad["Ad Related Info<br/>User id, Ad id"]
    profile["Seller profile<br/>Activity"]
    user -- "User id" --> ad
    ad --> profile
```

Useful metrics:

| Metric | Why useful |
| --- | --- |
| Ads in last 12 months | Captures seller volume. |
| Active months | Captures persistence. |
| Max concurrent live ads | Captures maintained inventory. |
| Recent listing activity | Keeps outreach focused on reachable sellers. |

Candidate rule: listings in 3+ months, 10+ ads in 12 months or multiple concurrent live ads, and at least one recent listing.


### 2. Paid Usage and Commercial Relationship

**Signal:** willingness to pay for visibility or Pro tools is one of the clearest business indicators.

Schemas: **Feature Related Info**, **Pro Info**, **Pro Invoicing Info**, linked through **User Related Info** and **Ad Related Info**.

```mermaid
flowchart LR
    user["User Related Info<br/>User id"]
    ad["Ad Related Info<br/>User id, Ad id"]
    feature["Feature Related Info<br/>Ad id"]
    pro["Pro Info<br/>User id"]
    invoice["Pro Invoicing Info<br/>User id"]
    profile["Seller profile<br/>Paid usage"]
    user -- "User id" --> ad
    ad -- "Ad id" --> feature
    user -- "User id" --> pro
    user -- "User id" --> invoice
    feature --> profile
    pro --> profile
    invoice --> profile
```

Useful metrics:

| Metric | Why useful |
| --- | --- |
| Paid feature count and spend | Shows paid visibility behavior. |
| Pro ads / CPC usage | Indicates use of the SMB proposition. |
| Invoice months and amount | Shows recurring commercial relationship. |
| Paid URL or external URL usage | Suggests traffic to an external business. |

Candidate rule: repeated paid feature usage, Pro activity, invoice history, or paid/external URL usage across multiple ads.


### 3. Category Specialization

**Signal:** SMBs often specialize in one category or a coherent category cluster.

Schemas: **Ad Related Info**, **Reference: Categories**.

```mermaid
flowchart LR
    ad["Ad Related Info<br/>User id, Ad category"]
    category["Reference: Categories<br/>Dutch + English labels"]
    profile["Seller profile<br/>Category specialization"]
    ad -- "Category" --> category
    ad -- "Group by User id" --> profile
    category --> profile
```

Useful metrics:

| Metric | Why useful |
| --- | --- |
| 2+ ads in same level-2 category within 90 days | Captures repeated specialized supply. |
| 5+ ads in same level-1 category over 12 months | Captures sustained category presence. |
| Share of ads in top category | Separates specialists from casual mixed sellers. |
| Business-prone categories | Adds category context from the assignment prompt. |

Candidate rule: repeated listings in one category, strong concentration, and preferably a business-prone category or related category cluster.


### 4. Buyer Engagement Aggregated to Seller Level

**Signal:** one popular ad can be consumer behavior; repeated buyer engagement across ads is more SMB-like.

Schemas: **Ad Related Info**, **Messaging**, **Traffic Info**.

```mermaid
flowchart LR
    ad["Ad Related Info<br/>User id, Ad id"]
    messaging["Messaging<br/>Ad id, Seller id, Buyer id"]
    traffic["Traffic Info<br/>Ad / click events"]
    profile["Seller profile<br/>Buyer engagement"]
    ad -- "Ad id" --> messaging
    ad -- "Ad page / Ad id" --> traffic
    messaging --> profile
    traffic --> profile
    ad -- "Group by User id" --> profile
```

Useful metrics:

| Metric | Why useful |
| --- | --- |
| Engaged ads | Avoids one-ad outliers. |
| Unique engaged buyers | Captures breadth of demand. |
| Months with engagement | Captures persistence. |
| Engagement rate | Controls for exposure where possible. |

Candidate rule: top-decile engagement within category mix, 5+ engaged ads in 12 months, engagement in 3+ months, and multiple unique buyers.


### Controls

- Normalize by category and exposure where possible.
- Require breadth and persistence to avoid one-off consumer outliers.
- Deduplicate buyers before counting demand.
- Treat every method as a likelihood signal, not a definitive label.


## Decision Mechanism: SMB Likelihood Score

Combine the four methods into one seller-level score. Each method contributes evidence; sellers with multiple independent signals should rank higher than sellers with only one signal.

Scoring principles:

| Principle | Rationale |
| --- | --- |
| Use weighted signals | Some signals are stronger than others, especially paid or verified business behavior. |
| Reward consistency | Repeated behavior across ads, months, and categories is stronger than a one-time spike. |
| Keep explainability | Every seller should have clear `top_reasons` behind the score. |
| Calibrate thresholds | Cutoffs should depend on campaign size, outreach cost, and acceptable false positives. |

The output should be a ranked seller list with `user_id`, `smb_score`, confidence tier, top reasons, and recommended action.


### If Known SMB Labels Exist

With known SMB labels, replace manual weights with a supervised model.

Use the same seller-level features from the four methods, train the model to predict SMB likelihood, and validate against business goals such as precision for sales outreach or recall for broad marketing.

Even with a model, keep the output explainable so commercial teams can understand why a seller was prioritized.
